# Autocodificadores Convolucionais para Remoção de Ruído (*Denoising*)

## Introdução

Um autocodificador é uma arquitetura de rede neural não supervisionada treinada para tentar copiar sua entrada para sua saída. Ele possui uma estrutura que segue um formato de "ampulheta":

1. **Encoder (Codificador)**: Comprime a entrada em uma representação de dimensão reduzida (espaço latente).
2. **Bottleneck (Gargalo)**: A representação compactada dos dados mais importantes.
3. **Decoder (Decodificador)**: Reconstrói a entrada original a partir da representação latente. 

<figure style="text-align: center;">
  <img src="assets/figura-1.png" alt="Figura 1" width="800">
</figure>

**Figura 1** – Diagrama arquitetural de um autocodificador tradicional. [<a href="#ref1">1</a>]

Em um *Denoising Autoencoder* (DAE), é corrompida intencionalmente a entrada com ruído (neste caso, gaussiano), mas forçamos a rede a prever a imagem original limpa. Isso impede que a rede apenas decore a entrada (como uma função identidade) e a obriga a aprender padrões geométricos robustos (como traços e curvas) para restaurar a informação perdida.

Sobre o DAE, podemos citar algumas aplicações relevantes:
 * Melhoria de imagens médicas (raio-x, ressonância) [<a href="#ref2">2</a>].
 * Pré-processamento para OCR  [<a href="#ref3">3</a>].
 * Detecção de anomalias (analisando o erro de reconstrução) [<a href="#ref4">4</a>].


## Preparação dos Dados

Utilizaremos o dataset MNIST (dígitos manuscritos). O processo envolve:

1. **Normalização**: Convertemos os pixels de `[0, 255]` para `[0,1]`. Isso é vital para a convergência do gradiente e para o uso da função de ativação `sigmoid` na saída.
2. **Reshape**: Adicionamos a dimensão de canal `(28, 28, 1)`, pois as camadas convolucionais esperam tensores 4D `(batch, altura, largura, canais)`.
3. **Ruído Aditivo Gaussiano**: Adicionamos valores aleatórios da distribuição normal.

$$
\tilde{x} = x + \mathcal{N}(0, \sigma)
$$

Onde o `noise_factor` atua como o desvio padrão. O uso do `np.clip` garante que, após somar o ruído, os pixels permaneçam no intervalo válido de imagem.

In [ ]:
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import utils
import os

import pandas as pd
import seaborn as sns

import plotly.io as pio
pio.renderers.default = "notebook"

# remove avisos do TensorFlow no console
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
# Configuração para garantir a reprodutibilidade
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
BATCH_SIZE = 128
EPOCHS = 15
NOISE_FACTOR = 0.5

In [ ]:
def load_and_corrupt_mnist(noise_factor=NOISE_FACTOR):
    """
    Carrega o dataset MNIST, aplica normalização [0, 1], ajusta as dimensões para 
    (H, W, C) e injeta ruido gaussiano.
    """

    (x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
    x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))

    x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
    x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)

    x_train_noisy = np.clip(x_train_noisy, 0., 1.)
    x_test_noisy = np.clip(x_test_noisy, 0., 1.)

    return (x_train_noisy, x_train), (x_test_noisy, x_test)

(x_train_noisy, x_train_clean), (x_test_noisy, x_test_clean) = load_and_corrupt_mnist()

In [ ]:
def display_noise_comparison(clean, noisy, n=5):
    # mostra as imagens com e sem ruído
    plt.figure(figsize=(10, 4))
    for i in range(n):
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(tf.squeeze(clean[i]), cmap="gray")
        plt.title("Original")
        plt.axis("off")

        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(tf.squeeze(noisy[i]), cmap="gray")
        plt.title("Entrada com Ruído")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

display_noise_comparison(x_test_clean, x_test_noisy)

## Arquitetura do Modelo: Convolutional Autoencoder

Ao contrário de redes densas (MLP), usamos camadas convolucionais para preservar a relação espacial dos pixels.

### O Codificador (Downsampling)

O objetivo é reduzir a dimensionalidade espacial enquanto aumentamos a profundidade (número de filtros/*features*). 

`Conv2D` com `Strides=2`: Ao invés de usar `MaxPooling` (que descarta informações), usamos convoluções com passo 2 para aprender a melhor forma de reduzir a imagem.

* **Compressão**: A entrada $28 \times 28 \times 1$ é reduzida para $7 \times 7 \times 64$.

### O Decodificador (Upsampling)

O objetivo é recuperar a dimensão espacial. **Conv2DTranspose**: Frequentemente chamada de deconvolução (termo tecnicamente impreciso, mas comum), esta operação aprende a fazer o *upsampling* dos mapas de características, preenchendo os detalhes faltantes.

### Saída e Função de Perda

* **Ativação Sigmoid**: Como normalizamos os dados entre [0, 1], a última camada deve usar sigmoide para garantir que a saída esteja na mesma escala.
* **Loss Binary Crossentropy**: Embora estejamos lidando com imagens (onde MSE é comum), tratar a intensidade do pixel como uma probabilidade de ser branco ou preto funciona bem para o MNIST, gerando gradientes mais fortes que o MSE nas bordas dos dígitos. O dataset MNIST é composto majoritariamente por pixels pretos (valor 0). Se o modelo "chutar" que a imagem inteira é preta (0), o erro MSE será muito baixo (pois ele acertou 80% da imagem que é fundo), o que provavelmente imagens puramente com valores 0 seria algo aceitável para essa métrica. A `binary_crossentropy`, por outro lado, penaliza severamente erros de confiança, forçando o modelo a captar os padrões para de fato desenhar o dígito.

In [ ]:
def build_denoising_autoencoder():
    # modelo arquitetural do autocodificador para remoção de ruído.
    
    input_img = layers.Input(shape=(28, 28, 1))

    # codificador 28x28 -> 14x14
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same", strides=2)(input_img)
    # 14x14 -> 7x7
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same", strides=2)(x)

    # criação do espaço latente
    x = layers.Conv2DTranspose(64, (3,3), strides=2, activation="relu", padding="same")(x)
    x = layers.Conv2DTranspose(32, (3,3), strides=2, activation="relu", padding="same")(x)

    # decodificador
    decoded = layers.Conv2D(1, (3,3), activation="sigmoid", padding="same")(x)

    autoencoder = models.Model(input_img, decoded, name="mnist_denoiser")

    return autoencoder

model = build_denoising_autoencoder()
model.summary()

**Contexto:**
Um autoencoder tradicional busca aprender uma representação dos dados que permita reconstruí-los, enquanto um autoencoder de remoção de ruído (denoising autoencoder) é treinado para reconstruir a entrada original a partir de uma versão corrompida.

Analise as dimensões do modelo:
1.  **Entrada:** $28 \times 28 \times 1 = \mathbf{784}$ valores (pixels).
2.  **Espaço Latente (Camada `conv2d_1`):** $7 \times 7 \times 64$ filtros.

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 1:</b><br>
  O número de valores no espaço latente é maior ou menor que o número de valores da entrada? Considerando que o espaço latente possui maior dimensionalidade, por que o modelo ainda pode aprender a remover o ruído em vez de simplesmente aprender uma transformação identidade, copiando a entrada pixel a pixel?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy")

In [ ]:
MODEL_DIR = "saved_models"
MODEL_NAME = "mnist_denoiser.keras"
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_NAME)

os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
callbacks_list = [
    callbacks.EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint("saved_models/mnist_denoiser.keras", save_best_only=True)
]

history = model.fit(
    x_train_noisy, x_train_clean,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    shuffle=True, validation_split=0.1,
    callbacks=callbacks_list
)

In [ ]:
# Carrega pesos previamente treinados e salvos
# model = tf.keras.models.load_model(MODEL_PATH)

In [ ]:
denoised_images = model.predict(x_test_noisy)

## Avaliações do Modelo

1. **SSIM (Structural Similarity Index Measure)**: Mede a similaridade perceptual (luminância, contraste e estrutura). Varia de -1 a 1. Quanto mais próximo de 1, melhor a preservação das estruturas do dígito. Resultados acima de 0.70 indica, alta fidelidade estrutural.
2. **PSNR (Peak Signal-to-Noise Ratio)**: Mede a relação entre a potência máxima possível do sinal e a potência do ruído que afeta a fidelidade. Medido em decibéis (dB), valores acima de 20dB geralmente são considerados aceitáveis para compressão/restauração de imagens simples.

$$
PSNR = 10.log_{10}(\frac{MAX_{I}^{2}}{MSE})
$$

In [ ]:
def display_results(noisy, denoised, clean, n=7):
    # mostra os resultados da remoção de ruído nos dados de teste
    plt.figure(figsize=(16, 6))
    for i in range(n):
        ax = plt.subplot(3, n, i+1)
        plt.imshow(tf.squeeze(noisy[i]), cmap="gray")
        plt.title("Entrada (com ruído)")
        plt.axis("off")

        ax = plt.subplot(3, n, i+1 + n)
        plt.imshow(tf.squeeze(denoised[i]), cmap="gray")
        plt.title("Com ruído removido")
        plt.axis("off")

        ax = plt.subplot(3, n, i+1 + 2*n)
        plt.imshow(tf.squeeze(clean[i]), cmap="gray")
        plt.title("Imagens Originais")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

display_results(x_test_noisy, denoised_images, x_test_clean)

**Contexto:**
Treinamos o modelo com um `NOISE_FACTOR = 0.5`. 

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 2:</b><br>
  O modelo conseguiria recuperar os dígitos com ruído 0.9? Teste e justifique a resposta usando as métricas SSIM e PSNR.
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [ ]:
ssim_scores = tf.image.ssim(x_test_clean, denoised_images, max_val=1.0)
print(f"Média SSIM: {np.mean(ssim_scores):.4f}")

In [ ]:
psnr_scores = tf.image.psnr(x_test_clean, denoised_images, max_val=1.0)
print(f"Média PSNR: {np.mean(psnr_scores):.2f} dB")

### Análise de Desempenho por Classe

O Boxplot abaixo revela nuances interessantes sobre o modelo:

* **Variabilidade**: Note que dígitos como o "1" tendem a ter um PSNR mais alto e menor variância (caixa menor). Isso ocorre porque o "1" tem geometria simples e muito espaço vazio (fundo preto), facilitando a reconstrução.
* **Dificuldade**: Dígitos mais complexos ou com curvas fechadas (como "8", "5" ou "2") podem apresentar menor PSNR ou mais *outliers*, pois o modelo pode não performar tão bem ao reconstruir intersecções ruidosas.

Isso demonstra que o desempenho do modelo não é uniforme e depende da complexidade topológica do dado de entrada.

In [ ]:
(_, _), (_, y_test) = tf.keras.datasets.mnist.load_data()

def plot_psnr_by_class(psnr_scores, labels):
    # plota um bocplot do PSNR calculado para cada um dos dígitos.
    
    df = pd.DataFrame({
        'PSNR': psnr_scores,
        'Dígito': labels
    })
    
    plt.figure(figsize=(12, 6))
    sns.boxplot(x="Dígito", y="PSNR", data=df, hue="Dígito", palette="viridis", legend=False)
    
    plt.title("Performance da Remoçao de Ruído por Classe (Dígito)", fontsize=14)
    plt.ylabel("PSNR (dB)")
    plt.xlabel("Classe Real")
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()

plot_psnr_by_class(psnr_scores, y_test)

### Visualização de Mapas de Características

Uma das discussões comuns na área de aprendizagem profunda, é o comportamento dos modelos como "caixas pretas". No entanto, em Redes Convolucionais (CNNs), podemos visualizar o que cada camada está gerando ao extrair os **Mapas de Características (Feature Maps)**.

Abaixo, vamos visualizar a resposta de cada filtro da rede para uma entrada específica. Observe o fluxo da informação:

1.  **Codificador (`conv2d` e `conv2d_1`):**
    * As primeiras camadas atuam como extratores de bordas e texturas.
    * À medida que a rede aprofunda, a resolução espacial diminui (de $28\times28$ para $14\times14$, depois $7\times7$), mas a profundidade de abstração aumenta (mais filtros).
    * Note como a representação torna-se abstrata e irreconhecível para o olho humano no gargalo (*bottleneck*), focando na presença de padrões geométricos em vez de pixels exatos.

2.  **Decodificador (`conv2d_transpose`):**
    * Aqui ocorre o processo inverso. A rede tenta gerar novamente a imagem baseada nas características comprimidas.
    * Os filtros aprendem a preencher o espaço, recuperando a forma do dígito e ignorando o ruído aleatório.

Desse modo, devemos observar dois detalhes do decodificador até a saída:
* **Pixels Brilhantes:** Indicam alta ativação (o filtro encontrou o padrão que procurava).
* **Quadrados Pretos/Escuros:** Indicam filtros não usados ou que não foram ativados para este dígito específico (esparsidade).

In [ ]:
test_idx = np.random.randint(0, len(x_test_noisy))
test_image = x_test_noisy[test_idx]

plt.figure(figsize=(6, 3))
ax = plt.subplot(1, 2, 1)
plt.imshow(test_image.squeeze(), cmap="gray")
plt.title("Entrada Ruidosa")
plt.axis("off")

ax = plt.subplot(1, 2, 2)
prediction = model.predict(np.expand_dims(test_image, axis=0), verbose=0)
plt.imshow(prediction.squeeze(), cmap="gray")
plt.title("Saída Final")
plt.axis("off")
plt.show()

# mostra uma visualização das imagens a partir do contexto interno do modelo
utils.visualize_feature_maps(model, test_image)
# DICA: rode várias vezes essa célula e veja diferentes exemplos de dígitos

### Visualização 3D do Fluxo de Tensores

A imagem abaixo ilustra o caminho que os dados percorrem dentro da rede neural, evidenciando a transformação dimensional passo a passo. 

Observe a estrutura clássica de **Ampulheta**:

1.  **Entrada ($28 \times 28$):** A imagem ruidosa entra na rede. Observe que o sinal (o dígito) está corrompido por pixelização aleatória.
2.  **Compressão (Codificador):**
    * A cada passo, a resolução espacial diminui ($28 \rightarrow 14 \rightarrow 7$).
    * Isso força a rede a descartar informações redundantes. Como o ruído é aleatório e não possui padrão estrutural, ele é a primeira informação a ser descartada durante a compressão.
3.  **Gargalo / Espaço Latente (Centro $7 \times 7$):**
    * A imagem original foi comprimida em uma representação abstrata de baixa dimensão.
    * Neste ponto, não existem mais representações próximas dos pixels enxergáveis, apenas **características** de alto nível que descrevem a essência do dígito.
4.  **Reconstrução (Decodificador):**
    * A rede expande a representação latente de volta para $14 \times 14$ e finalmente $28 \times 28$.
    * Note que a saída é uma versão suavizada da entrada. O ruído desapareceu, mas as bordas do dígito também ficaram levemente borradas (blur).

A ideia do DAE é que o modelo aprendeu o subespaço onde dígitos manuscritos existem e projeta qualquer entrada ruidosa de volta para esse subespaço limpo.

In [ ]:
test_idx = np.random.randint(0, len(x_test_noisy))
test_image = x_test_noisy[test_idx]

utils.visualize_architecture_final(model, test_image)
# DICA: rode várias vezes essa célula e veja diferentes exemplos de dígitos

### Análise do Espaço Latente

O nosso "bottleneck" ou gargalo é $7 \times 7 \times 64 = 3136$ parâmetros. A entrada é $28 \times 28 = 784$ parâmetros. 

Tecnicamente, portanto, a camada latente não é menor do que a entrada original, logo não constitui um *bottleneck* dimensional estrito. Em vez disso, o modelo impõe um *bottleneck* mais sutil, baseado em esparsidade e restrições arquiteturais: a estrutura convolucional limita o espaço de representações possíveis, e o ruído adicionado à entrada força o modelo a aprender características robustas, em vez de simplesmente memorizar a disposição dos pixels nas imagens.

É importante destacar que se este autocodificador fosse implementado com camadas densas e uma camada intermediária de maior dimensão que a entrada, o modelo tenderia a aprender a função identidade, falhando em remover o ruído. A convolução e a natureza local dos filtros são justamente o que impede essa cópia direta e permitem a extração de características relevantes.

**Contexto:**
Utilizamos a `binary_crossentropy` como função de perda. Ela trata cada pixel como uma probabilidade de ser preto ou branco. No entanto, em restauração de imagens, frequentemente queremos minimizar a distância euclidiana entre a cor do pixel real e o previsto.

**Tarefa:**
Altere a compilação do modelo para usar o Erro Quadrático Médio (MSE), que penaliza grandes desvios de forma mais agressiva.

**O que deve ser alterado no código:**
1.  Volte na célula onde definimos `model.compile`.
2.  Troque o parâmetro `loss`:
    * **De:** `model.compile(optimizer="adam", loss="binary_crossentropy")`
    * **Para:** `model.compile(optimizer="adam", loss="mse")`
3.  Retreine o modelo (pode reduzir para 5 épocas para ser rápido).

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 3:</b><br>
  Provavelmente, você observou que os dígitos desapareceram ou a imagem ficou quase totalmente preta/cinza. Por que isso acontece?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

## Referências

<p id="ref1">[1] The MathWorks, Inc. (s.d.). Autoencoders (Autocodificadores). Recuperado de https://la.mathworks.com/discovery/autoencoder.html.
<p id="ref2">[2] Gondara, L. (2016, December). Medical image denoising using convolutional denoising autoencoders. In 2016 IEEE 16th international conference on data mining workshops (ICDMW) (pp. 241-246). IEEE.
<p id="ref3">[3] Wiraatmaja, C., Gunadi, K., & Sandjaja, I. N. (2017, September). The application of deep convolutional denoising autoencoder for optical character recognition preprocessing. In 2017 International Conference on Soft Computing, Intelligent System and Information Technology (ICSIIT) (pp. 72-77). IEEE.
<p id="ref4">[4] Kascenas, A., Pugeault, N., & O’Neil, A. Q. (2022, December). Denoising autoencoders for unsupervised anomaly detection in brain MRI. In International Conference on Medical Imaging with Deep Learning (pp. 653-664). PMLR.